In [21]:
# Hyperparameters
DEVICE_ID = 1
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
OPTIMIZER = "AdamW"
LOG_DIR = f"./combined_logs/"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
CLASSIFIER_NAME = "convnext_nano.in12k"
CLASSIFIER_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/classification/Beemachine/convnext_nano.in12k_timm_new_dataset_logs/checkpoints/best_model.pth"

In [22]:
import os
import timm
import numpy as np
import pandas as pd
import random
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [23]:
# Load classifier
def load_classifier(name, weights, num_classes, device_id):
    # Define the classification model and load weights
    model = timm.create_model(name, pretrained=False, num_classes=num_classes).to(f"cuda:{device_id}")
    state_dict_c = torch.load(weights, map_location=f"cuda:{device_id}")
    model.load_state_dict(state_dict_c) # Load the classifier state dict
    return model

# --- Replace NaNs within each species group ---
def fill_by_class_mean(df, class_col="species"):
    df = df.replace(0, np.nan)
    df = df.dropna(axis=1, how='all')
    df_numeric = df.select_dtypes(include=[np.number])
    # Fill NaNs in numeric columns using the class-wise mean
    df[df_numeric.columns] = df.groupby(class_col)[df_numeric.columns].transform(
        lambda x: x.fillna(x.mean())
    )
    # Step 2: fill any remaining NaNs globally (for columns that were all NaN in a class)
    df[df_numeric.columns] = df[df_numeric.columns].fillna(df[df_numeric.columns].mean())
    return df

In [24]:
train_species_df = pd.read_csv(os.path.join(DATA_DIR, 'train_aug_labels.csv'))
val_species_df = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
test_species_df = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
# train_species_df.shape, val_species_df.shape, test_species_df.shape

train_feat_path = r"./predicted_shape_features/beemachine_partwhole_v5_train.csv"
val_feat_path   = r"./predicted_shape_features/beemachine_partwhole_v5_val.csv"
test_feat_path  = r"./predicted_shape_features/beemachine_partwhole_v5_test.csv"
train_feat_df = pd.read_csv(train_feat_path) 
val_feat_df = pd.read_csv(val_feat_path) 
test_feat_df = pd.read_csv(test_feat_path)
# train_feat_df.shape, val_feat_df.shape, test_feat_df.shape

train_df = pd.merge(train_feat_df, train_species_df, on='image')
val_df = pd.merge(val_feat_df, val_species_df, on='image')
test_df = pd.merge(test_feat_df, test_species_df, on='image')
train_df.shape, val_df.shape, test_df.shape

((34722, 939), (1158, 939), (771, 939))

In [26]:
train_df = fill_by_class_mean(train_df, class_col="species")
val_df   = fill_by_class_mean(val_df, class_col="species")
test_df = fill_by_class_mean(test_df, class_col="species")

X_train = train_df.drop(columns=["image", "species"])
X_val = val_df.drop(columns=["image", "species"])
X_test = test_df.drop(columns=["image", "species"])
y_train = train_df["species"]
y_val = val_df["species"]
y_test = test_df["species"]

# --- Sanity check ---
assert not X_train.isna().any().any(), "NaNs remain in X_train!"
assert not X_val.isna().any().any(), "NaNs remain in X_val!"
assert not X_test.isna().any().any(), "NaNs remain in X_val!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

✅ Class-wise NaN imputation complete — no missing values remain.


In [27]:
num_classes = len(set(y_train))
num_classes

160

In [28]:
# Loading timm model
device = torch.device(f"cuda:{DEVICE_ID}")
model = load_classifier(
            name=CLASSIFIER_NAME, 
            weights=CLASSIFIER_WEIGHTS, 
            num_classes=num_classes, 
            device_id=DEVICE_ID
        )
model.to(device)

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 80, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)
          (norm): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Conv2d(80, 320, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Conv2d(320, 80, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)


In [ ]:
# 1. Define the Combined Model
class MultiModalMLP(nn.Module):
    def __init__(self, n_tabular_features, visual_feature_dim=640, hidden_dim=256):
        super(MultiModalMLP, self).__init__()
        
        # Load ConvNeXt-Nano as a feature extractor
        # num_classes=0 removes the head and returns the pooled features
        self.visual_backbone = timm.create_model('convnext_nano', pretrained=True, num_classes=0)
        
        # The MLP head that takes (m + n) features
        combined_dim = visual_feature_dim + n_tabular_features
        
        self.mlp = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1) # Change to num_classes for classification
        )

    def forward(self, image, tabular_data):
        # Extract visual features (m)
        visual_features = self.visual_backbone(image) # Shape: [batch, 640]
        
        # Concatenate with CSV features (n)
        combined = torch.cat((visual_features, tabular_data), dim=1)
        
        # Pass through MLP
        return self.mlp(combined)

# 2. Preparation Logic
n_features = 10 # Example n from your CSV
model = MultiModalMLP(n_tabular_features=n_features)

# 3. Data Handling Tip
# Ensure your CSV features are normalized BEFORE passing to the model
scaler = StandardScaler()
# X_tabular_scaled = scaler.fit_transform(X_csv)